# 00 — Pipeline ETL Completo: Bronze → Silver → Gold

**Propósito**: Executar o pipeline completo de engenharia de dados para materializar as camadas Bronze, Silver e Gold a partir das fontes brutas.

**Pré-requisitos**:
- Credenciais AWS configuradas (`aws configure --profile mba-thesis`)
- Chaves de API no `.env` (para ingestão Bronze das APIs)
- Dependências Python instaladas (`pip install -r requirements.txt`)

**Saídas**:
- Camada Bronze: Respostas brutas de APIs e CSVs
- Camada Silver: Datasets normalizados, tipados e deduplicados
- Camada Gold: Agregações prontas para análise nos notebooks downstream (01-06)

---

## Visão Geral da Arquitetura

Este notebook implementa a **Arquitetura Medallion** (Databricks, 2023):

```
┌─────────────────────────────────────────────────────────────────┐
│  CAMADA BRONZE (Bruto)                                          │
│  • Respostas API IBGE (Censo 2010, 2022)                       │
│  • Portal da Transparência (transferências, sanções)           │
│  • Séries deflator IPCA/BCB                                    │
├─────────────────────────────────────────────────────────────────┤
│  CAMADA SILVER (Normalizada)                                    │
│  • Star schema com tabelas de dimensão + fato                  │
│  • Validação de tipos, deduplicação, deflação                │
│  • Código municipal IBGE padronizado (7 dígitos)              │
├─────────────────────────────────────────────────────────────────┤
│  CAMADA GOLD (Pronta para Análise)                             │
│  • Perfis socioeconômicos municipais consolidados              │
│  • Agregações estaduais                                         │
│  • Features para clusterização (normalizadas)                  │
│  • Datasets de análise para ML e estatística                    │
└─────────────────────────────────────────────────────────────────┘
```

**Notebooks Downstream**: Após executar este ETL, prossiga para:
- `01_exploratory_data_analysis.ipynb` — EDA nos datasets Gold
- `02_statistical_analysis.ipynb` — Regressão OLS
- `03_machine_learning.ipynb` — Modelos preditivos
- `04_clustering_analysis.ipynb` — Segmentação K-means
- `05_corrupcao_idh_clusters.ipynb` — Análise Corrupção vs IDH
- `06_pipeline_tese_completo.ipynb` — Notebook master da tese (reexecuta ETL + todas análises)


In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


---

## 1. Configuração do Ambiente

### 1.1 Instalar Dependências (se necessário)


In [ ]:
# Auto-install dependencies on first run
import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


### 1.2 Imports e Configuração


In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Project modules
from src.ingestion.ibge_client import IBGEIngestor
from src.ingestion.transparency_client import TransparencyIngestor
from src.processing.ibge_transformer import IBGETransformer
from src.processing.transparency_transformer import TransparencyTransformer
from src.processing.gold_transformer import GoldTransformer
from src.config.runtime_config import load_runtime_config

print(f"Python path: {(sys.path[0] if sys.path[0] == "." else "<project-root>")}")
print(f"Working directory: {Path.cwd().name if Path.cwd().name else "<root>"}")
print(f"Timestamp: {datetime.now().isoformat()}")


### 1.3 Configurações de Reprodutibilidade


In [ ]:
# Fixed seed for reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import random
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed fixed at {SEED}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


---

## 2. Camada Bronze — Ingestão de Dados Brutos

### 2.1 Metodologia

A camada Bronze preserva dados brutos exatamente como recebidos das fontes (Armbrust et al., 2020). Isso garante:
- **Auditabilidade**: Dados originais podem ser reexaminados
- **Reprodutibilidade**: Pipeline pode ser reexecutado do estado bruto
- **Compliance legal**: Requisitos LAI (2011) e LGPD (2018) para linhagem de dados

**Fontes de Dados**:
- API SIDRA IBGE: Dados municipais (Censo 2010, 2022)
- Portal da Transparência: Transferências federais e sanções
- BCB SGS: Série deflator IPCA

### 2.2 Carregar Configuração de Runtime


In [ ]:
# Carregar variaveis de ambiente do .env primeiro (maior prioridade)
from dotenv import load_dotenv
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"Ambiente carregado de {env_path}")
else:
    print("AVISO: arquivo .env nao encontrado")

# Carregar configuracao runtime (menor prioridade que .env)
config_path = Path("../config/runtime_config.json")
if config_path.exists():
    with open(config_path) as f:
        runtime_config = json.load(f)
else:
    runtime_config = {}

# Extrair configuracoes - .env tem precedencia
aws_profile = os.environ.get("AWS_PROFILE") or runtime_config.get("aws", {}).get("profile", "mba-thesis")
s3_bucket = os.environ.get("S3_BUCKET_NAME") or runtime_config.get("aws", {}).get("s3_bucket_name", "enok-mba-thesis-datalake")
use_local_cache = runtime_config.get("execution", {}).get("use_local_cache", True)

# Definir AWS_PROFILE para boto3 usa-lo
os.environ["AWS_PROFILE"] = aws_profile

print(f"Perfil AWS: {aws_profile}")
print(f"Bucket S3: {s3_bucket}")
print(f"Usar Cache Local: {use_local_cache}")


### 2.3 Executar Ingestão Bronze

**Nota**: Ingestão Bronze das APIs requer:
- IBGE: Sem chave de API (dados públicos)
- Portal da Transparência: `TRANSPARENCY_API_KEY` no `.env`

Para pular chamadas de API e usar dados locais existentes, configure:
```python
SKIP_BRONZE_INGESTION = True
```


In [ ]:
# ===== Camada Bronze: Sincronizacao S3 -> local =====
# Usa src.ingestion.s3_local_sync para espelhar s3://$S3_BUCKET/bronze/ em data/bronze/.
# Idempotente: baixa apenas arquivos ausentes ou com tamanho diferente.
from src.ingestion.s3_local_sync import sync_s3_prefix_to_local

local_data_dir = Path("../data")
bronze_dir = local_data_dir / "bronze"

s3_bucket = os.getenv("S3_BUCKET_NAME", "enok-mba-thesis-datalake")
print(f"Sincronizando s3://{s3_bucket}/bronze/ -> {bronze_dir}")

result = sync_s3_prefix_to_local(
    bucket=s3_bucket,
    prefix="bronze/",
    local_dir=bronze_dir,
)
print(f"[SYNC] baixados={result['downloaded']} ja_presentes={result['skipped']} erros={result['errors']}")

bronze_files = list(bronze_dir.rglob("*.json")) + list(bronze_dir.rglob("*.parquet"))
by_source = {}
for f in bronze_files:
    by_source.setdefault(f.parent.name, []).append(f)
print(f"\n[OK] Camada Bronze: {len(bronze_files)} arquivos no total")
for source, files in sorted(by_source.items()):
    print(f"  - {source}: {len(files)} arquivo(s)")


---

## 3. Camada Silver — Normalização e Transformação

### 3.1 Metodologia

Camada Silver aplica:
- **Schema enforcement**: Validação de tipos via `config/silver_schemas.json`
- **Deflação**: Nominal → Real BRL usando IPCA (base 2022)
- **Deduplicação**: Remove duplicatas de cargas incrementais
- **Star schema**: Tabelas de dimensão + fato (Kimball & Ross, 2013)

**Pipeline de Transformação**:
1. Carregar dados brutos Bronze
2. Aplicar mapeamentos de tipo e validações
3. Deflacionar valores monetários (IPCA)
4. Padronizar códigos municipais (IBGE 7 dígitos)
5. Escrever arquivos Parquet Silver


### 3.2 Executar Transformações Silver


In [ ]:
# ===== Camada Silver: Sincronizacao S3 -> local =====
# Se o Silver estiver ausente no S3, rode: bash scripts/02_silver_transformation.sh
silver_dir = local_data_dir / "silver"
print(f"Sincronizando s3://{s3_bucket}/silver/ -> {silver_dir}")

result = sync_s3_prefix_to_local(
    bucket=s3_bucket,
    prefix="silver/",
    local_dir=silver_dir,
)
print(f"[SYNC] baixados={result['downloaded']} ja_presentes={result['skipped']} erros={result['errors']}")

silver_files = list(silver_dir.rglob("*.parquet"))
if not silver_files:
    print("\n[AVISO] Silver ausente no S3. Rode: bash scripts/02_silver_transformation.sh")
else:
    datasets = {}
    for f in silver_files:
        datasets.setdefault(f.parent.name, []).append(f)
    print(f"\n[OK] Camada Silver: {len(silver_files)} arquivo(s) Parquet")
    for dataset, files in sorted(datasets.items()):
        print(f"  - {dataset}: {len(files)} arquivo(s)")


---

## 4. Camada Gold — Agregações Prontas para Análise

### 4.1 Metodologia

Camada Gold cria datasets desnormalizados e otimizados para análise (Kleppmann, 2017):
- **Perfis municipais**: Indicadores socioeconômicos + deltas (2010→2022)
- **Resumos estaduais**: Agregados por UF (27 estados)
- **Features de clusterização**: Vetores normalizados para K-means
- **Datasets de ML**: Datasets etiquetados para aprendizado supervisionado

**Principais Datasets Produzidos**:
| Dataset | Registros | Propósito |
|---------|-----------|-----------|
| `agg_municipality_socioeconomic` | ~5.570 | Vetores de features municipais |
| `agg_state_summary` | 27 | Agregações estaduais |
| `analysis_compliance` | 27 | Dataset pronto para ML estadual |
| `analysis_compliance_municipality` | ~5.570 | Dataset pronto para ML municipal |
| `consolidated_clustering` | ~5.565 | Features normalizadas para clusterização |


### 4.2 Executar Transformações Gold


In [ ]:
# ===== Camada Gold: Sincronizacao S3 -> local =====
# Se o Gold estiver ausente no S3, rode: bash scripts/03_gold_transformation.sh
gold_dir = local_data_dir / "gold"
print(f"Sincronizando s3://{s3_bucket}/gold/ -> {gold_dir}")

result = sync_s3_prefix_to_local(
    bucket=s3_bucket,
    prefix="gold/",
    local_dir=gold_dir,
)
print(f"[SYNC] baixados={result['downloaded']} ja_presentes={result['skipped']} erros={result['errors']}")

gold_files = list(gold_dir.rglob("*.parquet"))
if not gold_files:
    print("\n[AVISO] Gold ausente no S3. Rode: bash scripts/03_gold_transformation.sh")
else:
    datasets = {}
    for f in gold_files:
        datasets.setdefault(f.parent.name, []).append(f)
    print(f"\n[OK] Camada Gold: {len(gold_files)} arquivo(s) Parquet")
    for dataset, files in sorted(datasets.items()):
        print(f"  - {dataset}: {len(files)} arquivo(s)")
print("\n[OK] Dados Gold disponiveis - pronto para notebooks de analise (01-06)")


---

## 5. Validação ETL e Quality Checks

### 5.1 Contagens de Registros nas Camadas


In [ ]:
def count_layer_rows(layer_path: Path) -> dict:
    # Count rows in all Parquet files in a layer
    counts = {}
    if not layer_path.exists():
        return counts
    
    for parquet_file in layer_path.rglob("*.parquet"):
        try:
            df = pd.read_parquet(parquet_file)
            dataset = parquet_file.parent.name
            counts[dataset] = len(df)
        except Exception:
            pass
    return counts

bronze_counts = count_layer_rows(Path("../data/bronze"))
silver_counts = count_layer_rows(Path("../data/silver"))
gold_counts = count_layer_rows(Path("../data/gold"))

print("=" * 60)
print("ETL LAYER SUMMARY")
print("=" * 60)
print(f"\nBRONZE: {len(bronze_counts)} datasets, {sum(bronze_counts.values()):,} total rows")
print(f"SILVER: {len(silver_counts)} datasets, {sum(silver_counts.values()):,} total rows")
print(f"GOLD:   {len(gold_counts)} datasets, {sum(gold_counts.values()):,} total rows")

print("\n" + "=" * 60)
print("GOLD DATASETS (Downstream Analysis Ready)")
print("=" * 60)
for dataset, count in sorted(gold_counts.items()):
    print(f"  • {dataset}: {count:,} rows")


---

## 6. Próximos Passos

O pipeline ETL está completo. Datasets da camada Gold estão prontos para análise.

### Sequência Recomendada de Notebooks:

1. **`00_etl_pipeline.ipynb`** ← Você está aqui (ETL completo)
2. **`01_exploratory_data_analysis.ipynb`** — EDA, distribuições, quality checks
3. **`02_statistical_analysis.ipynb`** — Regressão OLS, correlações
4. **`03_machine_learning.ipynb`** — Modelos preditivos (ElasticNet, Random Forest)
5. **`04_clustering_analysis.ipynb`** — Segmentação K-means com PCA
6. **`05_corrupcao_idh_clusters.ipynb`** — Análise Corrupção vs IDH estratificada
7. **`06_pipeline_tese_completo.ipynb`** — Notebook master (reexecuta ETL + todas análises)

### Ou execute os scripts shell diretamente:

```bash
# Pipeline completo do zero (requer chaves de API)
./scripts/01_bronze_ingestion.sh
./scripts/02_silver_transformation.sh
./scripts/03_gold_transformation.sh

# Ou tudo de uma vez
./scripts/run_pipeline.sh
```

### Locais de Armazenamento:
- Local: `data/bronze/`, `data/silver/`, `data/gold/`
- S3: `s3://{s3_bucket}/bronze/`, `/silver/`, `/gold/`


---

## Referências

- **Armbrust, M. et al. (2020).** Lakehouse: A New Generation of Open Platforms. Databricks.
- **Databricks (2023).** Medallion Architecture: Best Practices.
- **Kimball, R.; Ross, M. (2013).** The Data Warehouse Toolkit. 3rd ed. Wiley.
- **Kleppmann, M. (2017).** Designing Data-Intensive Applications. O'Reilly.
- **Brasil (2011).** Lei nº 12.527/2011 — Lei de Acesso à Informação.
- **Brasil (2018).** Lei nº 13.709/2018 — LGPD.
